In [1]:
import os
import ast
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

# 加载 .env 文件中的环境变量
load_dotenv(r"D:\study\new\Large_language_module_besed\Agent\ReAct\.env", override=True)


class HelloAgentsLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("MODEL_NAME") or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("OPENAI_API_KEY")
        baseUrl = baseUrl or os.getenv("OPENAI_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 180))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"🧠 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("✅ 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"❌ 调用LLM API时发生错误: {e}")
            return None


In [2]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

In [3]:
class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        根据用户问题生成一个行动计划。
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        
        # 为了生成计划，我们构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]
        
        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整的计划
        response_text = self.llm_client.think(messages=messages) or ""
        
        print(f"✅ 计划已生成:\n{response_text}")
        
        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            print(f"原始响应: {response_text}")
            return []
        except Exception as e:
            print(f"❌ 解析计划时发生未知错误: {e}")
            return []


In [4]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决"当前步骤"，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对"当前步骤"的回答:
"""


In [5]:
class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        根据计划，逐步执行并解决问题。
        """
        history = "" # 用于存储历史步骤和结果的字符串
        
        print("\n--- 正在执行计划 ---")
        
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}: {step}")
            
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无", # 如果是第一步，则历史为空
                current_step=step
            )
            
            messages = [{"role": "user", "content": prompt}]
            
            response_text = self.llm_client.think(messages=messages) or ""
            
            # 更新历史记录，为下一步做准备
            history += f"步骤 {i+1}: {step}\n结果: {response_text}\n\n"
            
            print(f"✅ 步骤 {i+1} 已完成，结果: {response_text}")

        # 循环结束后，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer


In [6]:
class PlanAndSolveAgent:
    def __init__(self, llm_client):
        """
        初始化智能体，同时创建规划器和执行器实例。
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        运行智能体的完整流程:先规划，后执行。
        """
        print(f"\n--- 开始处理问题 ---\n问题: {question}")
        
        # 1. 调用规划器生成计划
        plan = self.planner.plan(question)
        
        # 检查计划是否成功生成
        if not plan:
            print("\n--- 任务终止 --- \n无法生成有效的行动计划。")
            return

        # 2. 调用执行器执行计划
        final_answer = self.executor.execute(question, plan)
        
        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")


In [7]:
# 创建 LLM 客户端
llmClient = HelloAgentsLLM()

# 创建并运行 PlanAndSolve 智能体
agent = PlanAndSolveAgent(llm_client=llmClient)
result = agent.run("请帮我规划一次为期三天的北京旅行，包括景点、交通和美食推荐。")

print("\n" + "=" * 60)
print("最终结果:")
print(result)
print("=" * 60)


--- 开始处理问题 ---
问题: 请帮我规划一次为期三天的北京旅行，包括景点、交通和美食推荐。
--- 正在生成计划 ---
🧠 正在调用 Qwen/Qwen3-32B 模型...
✅ 大语言模型响应成功:
```python
[
    "第一天：抵达北京后前往故宫博物院参观，提前在线预约门票",
    "上午游览天安门广场及人民英雄纪念碑，拍摄标志性建筑照片",
    "中午在故宫附近的四季民福（故宫店）品尝北京特色菜",
    "下午参观景山公园并登顶俯瞰故宫全景，步行至北海公园",
    "傍晚前往王府井步行街，体验商业区夜景并享用全聚德（前门店）烤鸭",
    "第二天：清晨乘旅游专线巴士前往八达岭长城，选择好汉坡徒步路线",
    "中午在长城脚下的特色餐厅品尝农家菜，返回市区后乘坐地铁至颐和园",
    "下午游览昆明湖、长廊和十七孔桥，租借园内自行车环湖骑行",
    "晚上在北三环附近的大董（工体店）预定包间享用精品烤鸭套餐",
    "第三天：上午乘地铁至南锣鼓巷，穿行烟袋斜街体验老北京胡同文化",
    "中午在胡大饭馆（簋街总店）品尝正宗北京炒肝和炸酱面",
    "下午参观798艺术区，依次游览尤伦斯当代艺术中心、草场地画廊等",
    "傍晚根据航班/高铁时间，选择前往首都国际机场或北京南站",
    "备选方案：若时间充裕，可增加国家博物馆（免费）或后海酒吧街行程"
]
```
✅ 计划已生成:
```python
[
    "第一天：抵达北京后前往故宫博物院参观，提前在线预约门票",
    "上午游览天安门广场及人民英雄纪念碑，拍摄标志性建筑照片",
    "中午在故宫附近的四季民福（故宫店）品尝北京特色菜",
    "下午参观景山公园并登顶俯瞰故宫全景，步行至北海公园",
    "傍晚前往王府井步行街，体验商业区夜景并享用全聚德（前门店）烤鸭",
    "第二天：清晨乘旅游专线巴士前往八达岭长城，选择好汉坡徒步路线",
    "中午在长城脚下的特色餐厅品尝农家菜，返回市区后乘坐地铁至颐和园",
    "下午游览昆明湖、长廊和十七孔桥，租借园内自行车环湖骑行",
    "晚上在北三环附近的大董（工体店）预定包间享用精品烤鸭套餐",
    "第三天：上午乘地铁至南锣鼓巷，穿行烟袋斜街体